# DeepFER — Facial Emotion Recognition using Two Machine Learning Models

This version replaces the CNN with **two traditional machine-learning classifiers**:
1. **Support Vector Machine (SVM)** — trained on HOG image features.
2. **Random Forest** — trained on the same HOG features.

Expected dataset structure:
```text
archive-3/
  train/
    angry/ ...
    happy/ ...
  test/
    angry/ ...
    happy/ ...
```
Change `DATASET_DIR` below if your dataset folder is elsewhere. HOG converts facial images into edge/shape descriptors before the classifiers learn emotion classes.

In [ ]:
import os
import time
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

In [ ]:
# Dataset paths
DATASET_DIR = "archive-3"  # Update this path if needed
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(TEST_DIR):
    raise FileNotFoundError(
        f"Could not find train/test folders inside '{DATASET_DIR}'. "
        "Update DATASET_DIR to the folder containing train and test."
    )

class_names = sorted(
    name for name in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, name))
)
if not class_names:
    raise ValueError("No emotion class folders found in the training directory.")

print("Emotion classes:", class_names)
print("Number of classes:", len(class_names))

In [ ]:
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def count_images(directory):
    return {
        label: sum(
            1 for filename in os.listdir(os.path.join(directory, label))
            if filename.lower().endswith(VALID_EXTENSIONS)
        )
        for label in class_names
        if os.path.isdir(os.path.join(directory, label))
    }

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("Training images:", train_counts)
print("Testing images:", test_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(train_counts.keys(), train_counts.values())
axes[0].set_title("Training image distribution")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(test_counts.keys(), test_counts.values())
axes[1].set_title("Testing image distribution")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 1. Image preprocessing and HOG feature extraction

- Resize each image to a small, consistent grayscale size to reduce computation.
- HOG (Histogram of Oriented Gradients) describes facial edges and shape patterns.
- The exact same preprocessing is used for training, testing, and single-image prediction.

In [ ]:
IMAGE_SIZE = (64, 64)

def extract_hog_features(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB").resize(IMAGE_SIZE)
        image = np.asarray(img, dtype=np.float32) / 255.0

    gray = rgb2gray(image)
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )
    return features.astype(np.float32)

def load_feature_dataset(directory):
    features, labels = [], []
    skipped = 0

    for label in class_names:
        class_dir = os.path.join(directory, label)
        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):
            if not filename.lower().endswith(VALID_EXTENSIONS):
                continue
            path = os.path.join(class_dir, filename)
            try:
                features.append(extract_hog_features(path))
                labels.append(label)
            except Exception as exc:
                skipped += 1
                print(f"Skipped unreadable image: {path} ({exc})")

    if not features:
        raise ValueError(f"No readable images found in {directory}")

    return np.asarray(features, dtype=np.float32), np.asarray(labels), skipped

start = time.time()
X_train, y_train, skipped_train = load_feature_dataset(TRAIN_DIR)
X_test, y_test, skipped_test = load_feature_dataset(TEST_DIR)

print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)
print("Skipped images:", skipped_train + skipped_test)
print(f"Feature extraction time: {(time.time() - start) / 60:.2f} minutes")

## 2. Model 1 — Support Vector Machine (SVM)

SVM with an RBF kernel can learn non-linear boundaries between HOG feature patterns. Feature scaling is fitted only on the training data using a pipeline.

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced"
    ))
])

start = time.time()
svm_model.fit(X_train, y_train)
svm_train_time = time.time() - start

svm_predictions = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_predictions)

print(f"SVM training time: {svm_train_time / 60:.2f} minutes")
print(f"SVM test accuracy: {svm_accuracy * 100:.2f}%")
print("\nSVM classification report:")
print(classification_report(
    y_test, svm_predictions, labels=class_names,
    target_names=class_names, zero_division=0
))

## 3. Model 2 — Random Forest

Random Forest combines many decision trees. It provides a second model for comparison with SVM.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    max_features="sqrt"
)

start = time.time()
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - start

rf_predictions = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)

print(f"Random Forest training time: {rf_train_time / 60:.2f} minutes")
print(f"Random Forest test accuracy: {rf_accuracy * 100:.2f}%")
print("\nRandom Forest classification report:")
print(classification_report(
    y_test, rf_predictions, labels=class_names,
    target_names=class_names, zero_division=0
))

## 4. Compare both models

Use the held-out test folder for the final comparison. Do not select hyperparameters based on test results; use a separate validation set if you want to tune the models.

In [ ]:
results = {
    "SVM": svm_accuracy * 100,
    "Random Forest": rf_accuracy * 100
}

plt.figure(figsize=(7, 5))
bars = plt.bar(results.keys(), results.values())
plt.ylabel("Test accuracy (%)")
plt.title("DeepFER: ML model comparison")
plt.ylim(0, 100)
for bar, value in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, value + 1,
             f"{value:.2f}%", ha="center")
plt.tight_layout()
plt.show()

print("Accuracy comparison:")
for name, value in results.items():
    print(f"{name}: {value:.2f}%")

In [ ]:
def plot_confusion(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.xlabel("Predicted emotion")
    plt.ylabel("Actual emotion")
    plt.title(f"{model_name} — Confusion Matrix")
    plt.tight_layout()
    plt.show()

plot_confusion(y_test, svm_predictions, "SVM")
plot_confusion(y_test, rf_predictions, "Random Forest")

## 5. Save both trained models

The saved `.joblib` files include the trained estimators. Keep the class names and preprocessing settings consistent when using them later.

In [ ]:
joblib.dump(svm_model, "DeepFER_SVM.joblib")
joblib.dump(rf_model, "DeepFER_RandomForest.joblib")
joblib.dump({
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "feature": "HOG",
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2)
}, "DeepFER_metadata.joblib")

print("Saved: DeepFER_SVM.joblib")
print("Saved: DeepFER_RandomForest.joblib")
print("Saved: DeepFER_metadata.joblib")

## 6. Predict emotion from a new image

Set `image_path` to a face image. The code displays predictions from both models. The SVM output here is a predicted class, not a calibrated confidence score.

In [ ]:
def predict_emotion(image_path):
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    features = extract_hog_features(image_path).reshape(1, -1)
    svm_prediction = svm_model.predict(features)[0]
    rf_prediction = rf_model.predict(features)[0]

    with Image.open(image_path) as img:
        plt.figure(figsize=(5, 5))
        plt.imshow(img.convert("RGB"))
        plt.axis("off")
        plt.title(
            f"SVM: {svm_prediction}\nRandom Forest: {rf_prediction}"
        )
        plt.show()

    print("SVM predicted emotion:", svm_prediction)
    print("Random Forest predicted emotion:", rf_prediction)
    return svm_prediction, rf_prediction

# Replace with the path to your test image:
image_path = r"C:\Users\Madhu\Desktop\Screenshot 2026-08-06 201113.png"
# svm_emotion, rf_emotion = predict_emotion(image_path)